# Week 2: Text Cleaning

Goal: standardize listing remarks so later steps (entity extraction, search) see consistent text.

1. Profile the raw remarks to see what needs cleaning
2. Show before/after examples for each cleaning step
3. Measure the improvement

Only short snippets of remarks are shown so the notebook is safe to commit.

In [1]:
import difflib
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from scripts.text_cleaning import ABBREVIATIONS, TextCleaner

cleaner = TextCleaner()
df = pd.read_csv(ROOT / "data/processed/listing_sample.csv")
df["remarks_clean"] = df["remarks"].apply(cleaner.clean_text)
print(f"{len(df)} remarks, {len(ABBREVIATIONS)} abbreviations in the dictionary")

1000 remarks, 49 abbreviations in the dictionary


## 1. Profiling report: what needs cleaning

In [2]:
profile = cleaner.profile_column(df, "remarks")

summary = pd.Series({
    "Null rate": f"{profile['null_rate']:.1%}",
    "Average length (characters)": f"{profile['avg_length']:.0f}",
    "Remarks with HTML": profile["has_html"],
    "Remarks with special characters": profile["has_non_ascii"],
    "Remarks with hidden zero-width characters": profile["has_zero_width"],
    "Remarks with line breaks": profile["has_line_breaks"],
    "Remarks with ALL CAPS words": profile["has_all_caps"],
    "Remarks with dollar amounts": profile["price_mentions"],
    "Remarks with square footage": profile["sqft_mentions"],
}, name="value")
summary.to_frame()

,value
Null rate,0.0%
Average length (characters),1344
Remarks with HTML,0
Remarks with special characters,349
Remarks with hidden zero-width characters,5
Remarks with line breaks,250
Remarks with ALL CAPS words,290
Remarks with dollar amounts,74
Remarks with square footage,385


In [3]:
pd.DataFrame(profile["common_abbreviations"], columns=["abbreviation", "count"])

,abbreviation,count
0,w/,49
1,a/c,33
2,ba,22
3,lvp,15
4,bd,14
5,hwy,13
6,approx,13
7,ave,12
8,br,9
9,blvd,8


In [4]:
pd.DataFrame(profile["common_terms"], columns=["most common phrases", "count"])

,most common phrases,count
0,primary suite,336
1,floor plan,330
2,living room,315
3,living space,307
4,car garage,298
5,home offers,280
6,natural light,268
7,sq ft,217
8,stainless steel,214
9,steel appliances,182


## 2. Before and after examples

One example per cleaning step, showing just the part of the remark that changed.

In [5]:
def changed_snippet(before, after, width=35):
    """Return the part of before/after around the first change."""
    sm = difflib.SequenceMatcher(None, before, after, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "equal":
            b = before[max(0, i1 - width): i2 + width]
            a = after[max(0, j1 - width): j2 + width]
            show = lambda s: s.replace("\r", "\\r").replace("\n", "\\n").replace("\t", "\\t")
            return show(b), show(a)
    return None

STEPS = [
    ("Unicode", cleaner.normalize_unicode),
    ("Whitespace / line breaks", cleaner.normalize_whitespace),
    ("ALL CAPS", cleaner.normalize_case),
    ("Punctuation", cleaner.normalize_punctuation),
    ("Number words", cleaner.normalize_number_words),
    ("Prices", cleaner.normalize_prices),
    ("Measurements", cleaner.normalize_measurements),
    ("Abbreviations", cleaner.expand_abbreviations),
]

rows = []
for name, step in STEPS:
    changed = 0
    example = None
    for text in df["remarks"].dropna():
        after = step(text)
        if after != text:
            changed += 1
            # For whitespace, pick an example with a line break so the change is visible
            if example is None and (step != cleaner.normalize_whitespace or "\n" in text):
                example = changed_snippet(text, after)
    rows.append({"step": name, "remarks changed": changed,
                 "before": example[0] if example else "", "after": example[1] if example else ""})

with pd.option_context("display.max_colwidth", 90):
    display(pd.DataFrame(rows).set_index("step"))

,remarks changed,before,after
step,,,
Unicode,340,shower. Freshly painted throughout — bright and move-in ready. The back,shower. Freshly painted throughout - bright and move-in ready. The back
Whitespace / line breaks,504,e for family gatherings and guests.\r\r\nThis home has been lovingly maintai,e for family gatherings and guests. This home has been lovingly maintai
ALL CAPS,257,"Picture yourself in this welcoming TURNKEY 5-bedroom, 4-bathroom CORNER home","Picture yourself in this welcoming turnkey 5-bedroom, 4-bathroom corner home"
Punctuation,124,"Lake with other rivers, and streams .","Lake with other rivers, and streams."
Number words,403,ghout with indoor laundry room and one bedroom and full bathroom downstai,ghout with indoor laundry room and 1 bedroom and full bathroom downstai
Prices,55,lly updated with approximately $300K in upgrades throughout such as cu,lly updated with approximately $300000 in upgrades throughout such as cu
Measurements,383,"room, 4-bathroom CORNER home with 2,520 sq ft. Highlights that make thi","room, 4-bathroom CORNER home with 2520 square feet. Highlights that ma"
Abbreviations,141,veniently located near San Gabriel Blvd with easy access to the 60 Free,veniently located near San Gabriel boulevard with easy access to the 60 F


### Cleaning user queries

The same cleaner works on search queries (these are our own sample queries, so we can show them in full).

In [6]:
queries = pd.read_csv(ROOT / "data/processed/sample_queries.csv")
queries["clean"] = queries["query"].apply(cleaner.clean_text)
changed = queries[queries["query"] != queries["clean"]][["query", "clean"]]
with pd.option_context("display.max_colwidth", 80):
    display(changed)

,query,clean
0,3 bedroom house with a pool in San Diego under $1.2M,3 bedroom house with a pool in San Diego under $1200000
6,fixer-upper under 700k in Sacramento,fixer-upper under 700000 in Sacramento
13,new construction homes under 900k in Temecula,new construction homes under 900000 in Temecula
55,estimate my monthly mortgage payment on an $850k home,estimate my monthly mortgage payment on an $850000 home


## 3. Did cleaning help?

Compare the raw and cleaned remarks on the problems found in the profile.

In [7]:
CHECKS = {
    "special characters": r"[^\x00-\x7f]",
    "hidden zero-width characters": "[\u200b-\u200f\ufeff]",
    "line breaks": r"[\r\n]",
    "ALL CAPS phrases (2+ words)": r"\b[A-Z]{3,} [A-Z]{2,}\b",
    "repeated !!": r"!!",
    "number words (three bedrooms)": r"\b(?:one|two|three|four|five|six)[\s-](?:bed|bath|car)",
    "non-standard sqft (sq ft, SF)": r"sq\.?\s?ft|sqft|\bSF\b",
}
before_after = pd.DataFrame({
    "before": {k: df["remarks"].str.contains(p, regex=True).sum() for k, p in CHECKS.items()},
    "after": {k: df["remarks_clean"].str.contains(p, regex=True).sum() for k, p in CHECKS.items()},
})
before_after

,before,after
special characters,349,26
hidden zero-width characters,5,0
line breaks,250,0
ALL CAPS phrases (2+ words),153,0
repeated !!,51,0
number words (three bedrooms),248,2
"non-standard sqft (sq ft, SF)",183,3


In [8]:
# Facts in one standard format are easier to extract in Week 3
STANDARD = {
    "bedroom count (3 bedroom)": r"\b\d+ (?:spacious |additional )?bedroom",
    "bathroom count (2.5 bathroom)": r"\b\d+(?:\.\d+)? (?:full )?bath",
    "square feet (2000 square feet)": r"\b\d+ square feet",
}
pd.DataFrame({
    "before": {k: df["remarks"].str.contains(p, case=False).sum() for k, p in STANDARD.items()},
    "after": {k: df["remarks_clean"].str.contains(p, case=False).sum() for k, p in STANDARD.items()},
})

,before,after
bedroom count (3 bedroom),297,449
bathroom count (2.5 bathroom),313,377
square feet (2000 square feet),155,388


## 4. Key findings

**I. What needed cleaning**

*No HTML, but lots of small formatting issues.*
- Special characters (curly quotes, dashes, bullets) in 35% of remarks
- Line breaks in 25%, ALL CAPS words in 29%
- Hidden zero-width characters in 5 remarks (invisible, but they break word matching)

**II. What cleaning fixed**

*899 of 1,000 remarks changed.*
- Line breaks, hidden characters, repeated "!!", and ALL CAPS phrases fully removed
- Number words like "three bedrooms" converted in 246 of 248 remarks
- Square footage formats standardized (183 remarks with "sq ft" or "SF" down to 3)

**III. Why it matters for Week 3**

*Key facts now appear in one standard format, which makes them easier to extract.*
- Remarks with a count like "3 bedroom" went from 297 to 449
- Remarks with "2000 square feet" format went from 155 to 388

**IV. Limits**
- Accented letters (é, ñ) are kept on purpose for place names and words like café
- "SF" can mean square feet or San Francisco, so it's only converted after a number
- The spec's price rule would turn "Residence 4M" into $4,000,000, so "M" only counts as millions after a "$"